In [1]:
import pandas as pd 

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [10]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

# Data preprocessing

In [11]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenization

In [ ]:
from transformers import (     #text to text transfer transformer , handles translation, summerization , que and ans 
    T5Tokenizer,                #text to number (35k tokens)
    Trainer,
    T5ForConditionalGeneration,
    TrainingArguments,
)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small") #raw data to tokens

NameError: name 'T5Tokenizer' is not defined

In [ ]:
#raw data to tokens and give this to input for fined tunning
def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding=max_length,max_length=512,truncate=True) #standardize number of tokens,,, padding is all word give this no. of length
    targets=tokenizer(data["summary"],padding=max_length,max_length=150,truncate=True)
    
    inputs['labels']= target['input_ids'] #token ids add to input as labels
    return inputs   # gives both input id and target id

In [ ]:
#appling tokenization train and validation dataset
train_dataset= train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

NameError: name 'train_data' is not defined

#Working with our model

In [ ]:
#nlp to generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small") #pretrained model


In [ ]:
import torch

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
#training arguments
training_args = TrainingArguments(
    output_dir="./results",          # multiple checkpoints will be saved in this directory
    num_train_epochs=6,              # number of training epochs
    weight_decay=0.01,                 # strength of weight decay
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=8,    # batch size for evaluation
    evaluation_strategy="epoch",     # evaluation strategy to adopt during training
    save_strategy="epoch",           # save strategy to adopt during training
    warmup_steps=500,                 # number of warmup steps for learning rate scheduler
)

In [ ]:
#trainer class provides an api for feature -complete training in pytorch, and it supports distributed training on multiple gpus and tpu's, trainer is used to fine tune the model on our dataset, it takes model, training arguments, training dataset and validation dataset as input
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,  # training arguments
    train_dataset=train_dataset,         # training dataset 
    eval_dataset=eval_dataset,           # evaluation dataset
)

In [ ]:
#train the model
trainer.train()

In [ ]:
# model load , fine tune and save the model for future use
model.save_pretrained("./saved_summarization_model")
tokenizer.save_pretrained("./saved_summarization_model")


In [ ]:
#this code is used to load the saved model and tokenizer for future use without retraining
model=T5ConditionalGeneration.from_pretrained("./saved_summarization_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summarization_model")

## Test the core logic of summerization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)#clean the input dialogue
    #TEXT TO TOKENS
    inputs = tokenizer(dialogue,padding="max_length", truncation=True, max_length=512,return_tensors="pt") #genrerate summery to tokens ids(encode)
    model.to(device)  # Move the model to the appropriate device
    targets = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], max_length=150, num_beams=4, early_stopping=True) 
    #generate summary using the model(decode)
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) #convert tokens to text
    return summary
    